# V_SCH_COURSE_OFFER Cleaning

Scope: clean and prepare course-offer availability rows already loaded in `df_raw`. This notebook does not access databases, credentials, parquet files, or external data. It creates in-memory pandas DataFrames only.


In [3]:
# Cell 1: copy raw DataFrame and standardize column names
import pandas as pd

from src.cleaning_utils import integer_like_report, normalize_id_columns

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)

TABLE_NAME = "V_SCH_COURSE_OFFER"
df_raw=pd.read_parquet(r"D:\AI\Real projects\Academic_Advisor\data\raw\v_sch_course_offers.parquet")
assert "df_raw" in globals(), "df_raw must already be loaded before running this notebook."


def normalize_column_names(columns: pd.Index) -> pd.Index:
    normalized = (
        pd.Index(columns)
        .astype("string")
        .str.strip()
        .str.lower()
        .str.replace(r"[^0-9a-z]+", "_", regex=True)
        .str.strip("_")
    )
    assert not normalized.duplicated().any(), "Column-name normalization created duplicate column names."
    return normalized


def build_null_report(frame: pd.DataFrame) -> pd.DataFrame:
    row_count = len(frame)
    null_count = frame.isna().sum()
    return (
        pd.DataFrame(
            {
                "column": frame.columns,
                "dtype": [str(dtype) for dtype in frame.dtypes],
                "null_count": [int(null_count[column]) for column in frame.columns],
                "null_percent": [
                    round(float(null_count[column] / row_count * 100), 4) if row_count else 0.0
                    for column in frame.columns
                ],
            }
        )
        .sort_values(["null_count", "column"], ascending=[False, True])
        .reset_index(drop=True)
    )


def clean_text_series(series: pd.Series) -> pd.Series:
    cleaned = series.astype("string").str.strip()
    empty_like = cleaned.eq("") | cleaned.str.lower().isin(["nan", "none", "null", "<na>"])
    return cleaned.mask(empty_like, pd.NA)


def to_nullable_int(series: pd.Series, column_name: str) -> pd.Series:
    numeric = pd.to_numeric(series, errors="coerce")
    non_numeric_mask = series.notna() & numeric.isna()
    fractional_mask = numeric.notna() & ((numeric % 1) != 0)

    assert not non_numeric_mask.any(), f"{column_name} contains non-numeric values."
    assert not fractional_mask.any(), f"{column_name} contains fractional/suffix values; cannot safely cast to Int64."

    return numeric.astype("Int64")


def build_id_issue_sample(frame: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    samples = []
    for column in columns:
        numeric = pd.to_numeric(frame[column], errors="coerce")
        non_numeric_mask = frame[column].notna() & numeric.isna()
        fractional_mask = numeric.notna() & ((numeric % 1) != 0)
        issue_values = frame.loc[non_numeric_mask | fractional_mask, column].drop_duplicates().head(25)
        if len(issue_values):
            samples.append(
                pd.DataFrame(
                    {
                        "column": column,
                        "value": issue_values.astype("string").to_list(),
                        "issue": "non_integer_or_fractional_value",
                    }
                )
            )
    return pd.concat(samples, ignore_index=True) if samples else pd.DataFrame(columns=["column", "value", "issue"])


df = df_raw.copy()
raw_shape = df.shape
raw_row_count = len(df)
df.columns = normalize_column_names(df.columns)
df["_source_row_order"] = range(len(df))

required_input_columns = [
    "level_category_id",
    "part_id",
    "department_id",
    "faculty_id",
    "course_id",
    "course_type_id",
    "course_name_sl",
    "faculty_name_sl",
    "department_name_sl",
    "course_credits",
    "allow_register",
]

missing_input_columns = sorted(set(required_input_columns) - set(df.columns))
extra_input_columns = sorted(set(df.columns) - set(required_input_columns) - {"_source_row_order"})

print("Table:", TABLE_NAME)
print("Raw shape:", raw_shape)
print("Working shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Missing input columns:", missing_input_columns)
print("Extra input columns:", extra_input_columns)

assert not missing_input_columns, f"Missing required input columns: {missing_input_columns}"


Table: V_SCH_COURSE_OFFER
Raw shape: (164737, 12)
Working shape: (164737, 13)
Columns: ['level_category_id', 'part_id', 'department_id', 'faculty_id', 'course_id', 'course_type_id', 'course_name_sl', 'faculty_name_sl', 'department_name_sl', 'course_credits', 'course_status', 'allow_register', '_source_row_order']
Missing input columns: []
Extra input columns: ['course_status']


In [4]:
# Cell 2: basic reports before cleaning
null_report_before_cleaning = build_null_report(df[required_input_columns])
allow_register_raw_value_counts = df["allow_register"].value_counts(dropna=False)
sample_rows_before_cleaning = df[required_input_columns].head(10).copy()

print("DataFrame info before cleaning:")
df[required_input_columns].info()

print("\nNull report before cleaning:")
display(null_report_before_cleaning)

print("\nallow_register value counts before cleaning:")
display(allow_register_raw_value_counts)

print("\nSample rows before cleaning:")
display(sample_rows_before_cleaning)


DataFrame info before cleaning:
<class 'pandas.DataFrame'>
RangeIndex: 164737 entries, 0 to 164736
Data columns (total 11 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   level_category_id   164737 non-null  float64
 1   part_id             164737 non-null  float64
 2   department_id       164737 non-null  float64
 3   faculty_id          164737 non-null  float64
 4   course_id           164737 non-null  float64
 5   course_type_id      164737 non-null  float64
 6   course_name_sl      164737 non-null  str    
 7   faculty_name_sl     164737 non-null  str    
 8   department_name_sl  164737 non-null  str    
 9   course_credits      164737 non-null  float64
 10  allow_register      33488 non-null   str    
dtypes: float64(7), str(4)
memory usage: 34.5 MB

Null report before cleaning:


,column,dtype,null_count,null_percent
0,allow_register,str,131249,79.6718
1,course_credits,float64,0,0.0000
2,course_id,float64,0,0.0000
3,course_name_sl,str,0,0.0000
4,course_type_id,float64,0,0.0000
5,department_id,float64,0,0.0000
6,department_name_sl,str,0,0.0000
7,faculty_id,float64,0,0.0000
8,faculty_name_sl,str,0,0.0000
9,level_category_id,float64,0,0.0000



allow_register value counts before cleaning:


allow_register
NaN    131249
Y       30134
N        3354
Name: count, dtype: int64


Sample rows before cleaning:


,level_category_id,part_id,department_id,faculty_id,course_id,course_type_id,course_name_sl,faculty_name_sl,department_name_sl,course_credits,allow_register
0,1.0,20050.0,40.111,7.111,1.111,1.0,مبادئ المحاسبة 1,كلية إدارة الأعمال,قسم المحاسبة و التدقيق,2.0,NaN
1,1.0,20051.0,40.111,7.111,1.111,1.0,مبادئ المحاسبة 1,كلية إدارة الأعمال,قسم المحاسبة و التدقيق,2.0,NaN
2,1.0,20052.0,40.111,7.111,1.111,1.0,مبادئ المحاسبة 1,كلية إدارة الأعمال,قسم المحاسبة و التدقيق,2.0,NaN
3,1.0,20053.0,40.111,7.111,1.111,1.0,مبادئ المحاسبة 1,كلية إدارة الأعمال,قسم المحاسبة و التدقيق,2.0,NaN
4,1.0,20060.0,40.111,7.111,1.111,1.0,مبادئ المحاسبة 1,كلية إدارة الأعمال,قسم المحاسبة و التدقيق,2.0,NaN
5,1.0,20061.0,40.111,7.111,1.111,1.0,مبادئ المحاسبة 1,كلية إدارة الأعمال,قسم المحاسبة و التدقيق,2.0,NaN
6,1.0,20062.0,40.111,7.111,1.111,1.0,مبادئ المحاسبة 1,كلية إدارة الأعمال,قسم المحاسبة و التدقيق,2.0,NaN
7,1.0,20063.0,40.111,7.111,1.111,1.0,مبادئ المحاسبة 1,كلية إدارة الأعمال,قسم المحاسبة و التدقيق,2.0,NaN
8,1.0,20070.0,40.111,7.111,1.111,1.0,مبادئ المحاسبة 1,كلية إدارة الأعمال,قسم المحاسبة و التدقيق,2.0,NaN
9,1.0,20071.0,40.111,7.111,1.111,1.0,مبادئ المحاسبة 1,كلية إدارة الأعمال,قسم المحاسبة و التدقيق,2.0,NaN


In [5]:
# Cell 3: clean strings
text_columns = [
    "course_name_sl",
    "faculty_name_sl",
    "department_name_sl",
    "allow_register",
]

for column in text_columns:
    df[column] = clean_text_series(df[column])

df["allow_register"] = df["allow_register"].str.upper()

string_cleaning_report = pd.DataFrame(
    {
        "column": text_columns,
        "dtype_after_cleaning": [str(df[column].dtype) for column in text_columns],
        "null_count_after_cleaning": [int(df[column].isna().sum()) for column in text_columns],
    }
)

print("String cleaning complete.")
display(string_cleaning_report)
print("allow_register value counts after string cleaning:")
display(df["allow_register"].value_counts(dropna=False))


String cleaning complete.


,column,dtype_after_cleaning,null_count_after_cleaning
0,course_name_sl,string,0
1,faculty_name_sl,string,0
2,department_name_sl,string,0
3,allow_register,string,131249


allow_register value counts after string cleaning:


allow_register
<NA>    131249
Y        30134
N         3354
Name: count, dtype: int64[pyarrow]

In [6]:
# Cell 4: validate allow_register
valid_allow_register_values = {"Y", "N"}

invalid_allow_register_mask = df["allow_register"].notna() & ~df["allow_register"].isin(valid_allow_register_values)
invalid_allow_register_report = df.loc[invalid_allow_register_mask].copy()

print("Invalid allow_register rows:", len(invalid_allow_register_report))
if len(invalid_allow_register_report):
    display(invalid_allow_register_report)
else:
    print("allow_register contains only Y, N, or missing values.")

assert invalid_allow_register_report.empty, "Unexpected allow_register values found. See invalid_allow_register_report."


Invalid allow_register rows: 0
allow_register contains only Y, N, or missing values.


In [7]:
# Cell 5: create registration availability flag
# Y means available. N and missing values mean unavailable.
df["is_registration_allowed"] = df["allow_register"].eq("Y").fillna(False).astype("boolean")

is_registration_allowed_distribution = df["is_registration_allowed"].value_counts(dropna=False)
allow_register_flag_crosscheck = pd.crosstab(
    df["allow_register"].fillna("<NA>"),
    df["is_registration_allowed"],
    dropna=False,
)

print("is_registration_allowed distribution:")
display(is_registration_allowed_distribution)

print("allow_register vs is_registration_allowed:")
display(allow_register_flag_crosscheck)

assert df["is_registration_allowed"].notna().all(), "is_registration_allowed must not contain null values."
assert df.loc[df["allow_register"].eq("Y"), "is_registration_allowed"].all(), "Y rows must be allowed."
assert not df.loc[df["allow_register"].ne("Y") | df["allow_register"].isna(), "is_registration_allowed"].any(), "N/missing rows must be unavailable."


is_registration_allowed distribution:


is_registration_allowed
False    134603
True      30134
Name: count, dtype: Int64

allow_register vs is_registration_allowed:


is_registration_allowed,False,True
allow_register,,
<NA>,131249,0
N,3354,0
Y,0,30134


In [8]:
# Cell 6: ID handling with project-approved helper functions
simple_integer_id_columns = [
    "level_category_id",
    "part_id",
    "course_type_id",
]

suffix_sensitive_id_columns = [
    "department_id",
    "faculty_id",
    "course_id",
]

all_id_columns = simple_integer_id_columns + suffix_sensitive_id_columns

id_validation_report = pd.DataFrame([integer_like_report(df, column) for column in all_id_columns])
id_issue_sample_report = build_id_issue_sample(df, all_id_columns)

print("ID validation report before conversion:")
display(id_validation_report)

print("ID non-integer/fractional value samples before conversion:")
display(id_issue_sample_report)

# These IDs are academic/category codes and are safe to cast only after validation rejects suffix/fractional values.
for column in simple_integer_id_columns:
    df[column] = to_nullable_int(df[column], column)

# These IDs can carry meaningful decimal suffixes such as .111, so preserve them as normalized strings.
df = normalize_id_columns(df, suffix_sensitive_id_columns)

id_dtype_report_after_cleaning = pd.DataFrame(
    {
        "column": all_id_columns,
        "dtype_after_cleaning": [str(df[column].dtype) for column in all_id_columns],
        "null_count_after_cleaning": [int(df[column].isna().sum()) for column in all_id_columns],
    }
)

print("ID dtypes after cleaning:")
display(id_dtype_report_after_cleaning)


ID validation report before conversion:


,column,source_dtype,non_null_count,numeric_count,non_numeric_or_null_count,fractional_count,fractional_ratio
0,level_category_id,float64,164737,164737,0,0,0.0
1,part_id,float64,164737,164737,0,0,0.0
2,course_type_id,float64,164737,164737,0,0,0.0
3,department_id,float64,164737,164737,0,164737,1.0
4,faculty_id,float64,164737,164737,0,164737,1.0
5,course_id,float64,164737,164737,0,164737,1.0


ID non-integer/fractional value samples before conversion:


,column,value,issue
0,department_id,40.111,non_integer_or_fractional_value
1,department_id,13.111,non_integer_or_fractional_value
2,department_id,180.111,non_integer_or_fractional_value
3,department_id,37.111,non_integer_or_fractional_value
4,department_id,179.111,non_integer_or_fractional_value
5,department_id,36.111,non_integer_or_fractional_value
6,department_id,38.111,non_integer_or_fractional_value
7,department_id,178.111,non_integer_or_fractional_value
8,department_id,35.111,non_integer_or_fractional_value
9,department_id,9.111,non_integer_or_fractional_value


ID dtypes after cleaning:


,column,dtype_after_cleaning,null_count_after_cleaning
0,level_category_id,Int64,0
1,part_id,Int64,0
2,course_type_id,Int64,0
3,department_id,string,0
4,faculty_id,string,0
5,course_id,string,0


In [9]:
# Cell 7: numeric cleaning for course_credits
course_credits_numeric = pd.to_numeric(df["course_credits"], errors="coerce")
invalid_course_credits_mask = df["course_credits"].notna() & course_credits_numeric.isna()
invalid_course_credits_report = df.loc[invalid_course_credits_mask].copy()

# Fractional credits are valid and must be preserved.
fractional_course_credits_mask = course_credits_numeric.notna() & ((course_credits_numeric % 1) != 0)
fractional_course_credits_report = df.loc[fractional_course_credits_mask].copy()

df["course_credits"] = course_credits_numeric.astype("Float64")

course_credits_describe = df["course_credits"].describe()
course_credits_value_counts = df["course_credits"].value_counts(dropna=False).sort_index()

print("Invalid course_credits rows coerced to NaN:", len(invalid_course_credits_report))
if len(invalid_course_credits_report):
    display(invalid_course_credits_report)

print("course_credits describe:")
display(course_credits_describe)

print("course_credits value counts:")
display(course_credits_value_counts)

print("Fractional course_credits rows:", len(fractional_course_credits_report))
display(fractional_course_credits_report.head(50))


Invalid course_credits rows coerced to NaN: 0
course_credits describe:


count    164737.0
mean     2.873192
std      1.168304
min           0.0
25%           2.0
50%           3.0
75%           3.0
max          24.0
Name: course_credits, dtype: Float64

course_credits value counts:


course_credits
0.0       1485
1.0       5742
2.0      35344
3.0     103455
4.0      13662
4.5        396
5.0       2772
6.0       1584
20.0        99
24.0       198
Name: count, dtype: Int64

Fractional course_credits rows: 396


,level_category_id,part_id,department_id,faculty_id,course_id,course_type_id,course_name_sl,faculty_name_sl,department_name_sl,course_credits,course_status,allow_register,_source_row_order,is_registration_allowed
69498,4,20050,23.111,2.111,704.111,3,أمراض الأطفال (1),كلية الطب البشري,قسم الأطفال,4.5,NaN,<NA>,69498,False
69499,4,20051,23.111,2.111,704.111,3,أمراض الأطفال (1),كلية الطب البشري,قسم الأطفال,4.5,NaN,<NA>,69499,False
69500,4,20052,23.111,2.111,704.111,3,أمراض الأطفال (1),كلية الطب البشري,قسم الأطفال,4.5,NaN,<NA>,69500,False
69501,4,20053,23.111,2.111,704.111,3,أمراض الأطفال (1),كلية الطب البشري,قسم الأطفال,4.5,NaN,<NA>,69501,False
69502,4,20060,23.111,2.111,704.111,3,أمراض الأطفال (1),كلية الطب البشري,قسم الأطفال,4.5,NaN,<NA>,69502,False
69503,4,20061,23.111,2.111,704.111,3,أمراض الأطفال (1),كلية الطب البشري,قسم الأطفال,4.5,NaN,<NA>,69503,False
69504,4,20062,23.111,2.111,704.111,3,أمراض الأطفال (1),كلية الطب البشري,قسم الأطفال,4.5,NaN,<NA>,69504,False
69505,4,20063,23.111,2.111,704.111,3,أمراض الأطفال (1),كلية الطب البشري,قسم الأطفال,4.5,NaN,<NA>,69505,False
69506,4,20070,23.111,2.111,704.111,3,أمراض الأطفال (1),كلية الطب البشري,قسم الأطفال,4.5,NaN,<NA>,69506,False
69507,4,20071,23.111,2.111,704.111,3,أمراض الأطفال (1),كلية الطب البشري,قسم الأطفال,4.5,NaN,<NA>,69507,False


In [10]:
# Cell 8: duplicate checks and duplicate removal
logical_key = [
    "part_id",
    "faculty_id",
    "department_id",
    "level_category_id",
    "course_id",
]

extended_key = [
    "part_id",
    "faculty_id",
    "department_id",
    "level_category_id",
    "course_id",
    "course_type_id",
]

duplicate_logical_key_report = (
    df.loc[df.duplicated(logical_key, keep=False)]
    .sort_values(logical_key + ["_source_row_order"])
    .copy()
)

duplicate_extended_key_report = (
    df.loc[df.duplicated(extended_key, keep=False)]
    .sort_values(extended_key + ["_source_row_order"])
    .copy()
)

exact_duplicate_compare_columns = [column for column in df.columns if column != "_source_row_order"]
exact_duplicate_report = df.loc[df.duplicated(exact_duplicate_compare_columns, keep=False)].copy()
exact_duplicate_rows_removed = int(df.duplicated(exact_duplicate_compare_columns, keep="first").sum())

print("Duplicate rows on logical key:", len(duplicate_logical_key_report))
print("Duplicate rows on extended key:", len(duplicate_extended_key_report))
print("Exact duplicate rows that will be removed:", exact_duplicate_rows_removed)

display(duplicate_logical_key_report)
display(duplicate_extended_key_report)

rows_before_duplicate_removal = len(df)
df = df.drop_duplicates(subset=exact_duplicate_compare_columns).copy()

# For remaining non-exact logical-key duplicates, keep highest registration priority: Y > N > missing.
df["_allow_priority"] = df["allow_register"].map({"Y": 2, "N": 1}).fillna(0).astype("int64")
rows_before_priority_dedup = len(df)

df = (
    df.sort_values(
        logical_key + ["_allow_priority", "_source_row_order"],
        ascending=[True] * len(logical_key) + [False, True],
    )
    .drop_duplicates(subset=logical_key, keep="first")
    .sort_values("_source_row_order")
    .drop(columns=["_allow_priority"])
    .reset_index(drop=True)
)

priority_duplicate_rows_removed = rows_before_priority_dedup - len(df)
duplicate_rows_removed = rows_before_duplicate_removal - len(df)

duplicate_removal_summary = pd.DataFrame(
    {
        "metric": [
            "rows_before_duplicate_removal",
            "exact_duplicate_rows_removed",
            "priority_duplicate_rows_removed",
            "total_duplicate_rows_removed",
            "rows_after_duplicate_removal",
        ],
        "value": [
            rows_before_duplicate_removal,
            exact_duplicate_rows_removed,
            priority_duplicate_rows_removed,
            duplicate_rows_removed,
            len(df),
        ],
    }
)

display(duplicate_removal_summary)


Duplicate rows on logical key: 2
Duplicate rows on extended key: 2
Exact duplicate rows that will be removed: 1


,level_category_id,part_id,department_id,faculty_id,course_id,course_type_id,course_name_sl,faculty_name_sl,department_name_sl,course_credits,course_status,allow_register,_source_row_order,is_registration_allowed
100556,1,20212,35.111,34.111,1017.111,1,لغة اجنبية حية (ثانية),وحدة متطلبات الجامعة,قسم متطلبات الجامعة,2.0,Y,Y,100556,True
100557,1,20212,35.111,34.111,1017.111,1,لغة اجنبية حية (ثانية),وحدة متطلبات الجامعة,قسم متطلبات الجامعة,2.0,Y,Y,100557,True


,level_category_id,part_id,department_id,faculty_id,course_id,course_type_id,course_name_sl,faculty_name_sl,department_name_sl,course_credits,course_status,allow_register,_source_row_order,is_registration_allowed
100556,1,20212,35.111,34.111,1017.111,1,لغة اجنبية حية (ثانية),وحدة متطلبات الجامعة,قسم متطلبات الجامعة,2.0,Y,Y,100556,True
100557,1,20212,35.111,34.111,1017.111,1,لغة اجنبية حية (ثانية),وحدة متطلبات الجامعة,قسم متطلبات الجامعة,2.0,Y,Y,100557,True


,metric,value
0,rows_before_duplicate_removal,164737
1,exact_duplicate_rows_removed,1
2,priority_duplicate_rows_removed,0
3,total_duplicate_rows_removed,1
4,rows_after_duplicate_removal,164736


In [11]:
# Cell 9: final validation
critical_non_allow_register_columns = [
    "level_category_id",
    "part_id",
    "department_id",
    "faculty_id",
    "course_id",
    "course_type_id",
    "course_name_sl",
    "faculty_name_sl",
    "department_name_sl",
    "course_credits",
]

final_business_columns = [
    "part_id",
    "faculty_id",
    "department_id",
    "level_category_id",
    "course_id",
    "course_type_id",
    "course_credits",
    "allow_register",
    "is_registration_allowed",
    "course_name_sl",
    "faculty_name_sl",
    "department_name_sl",
]

null_report_after_cleaning = build_null_report(df[final_business_columns])
bad_null_rows = df.loc[df[critical_non_allow_register_columns].isna().any(axis=1)].copy()
remaining_invalid_allow_register = df.loc[
    df["allow_register"].notna() & ~df["allow_register"].isin(valid_allow_register_values)
].copy()
missing_final_columns = sorted(set(final_business_columns) - set(df.columns))

print("Null report after cleaning:")
display(null_report_after_cleaning)

print("Bad null rows in critical non-allow_register columns:", len(bad_null_rows))
if len(bad_null_rows):
    display(bad_null_rows)

assert not missing_final_columns, f"Missing final columns: {missing_final_columns}"
assert bad_null_rows.empty, "Critical non-allow_register columns contain nulls. See bad_null_rows."
assert remaining_invalid_allow_register.empty, "Unexpected allow_register values remain."
assert not df.duplicated(logical_key).any(), "Duplicate logical keys remain after duplicate handling."
assert df["is_registration_allowed"].notna().all(), "is_registration_allowed contains nulls."
assert pd.api.types.is_numeric_dtype(df["course_credits"]), "course_credits must be numeric."

validation_summary = pd.DataFrame(
    {
        "metric": [
            "raw_rows",
            "clean_rows_before_final_column_selection",
            "raw_columns",
            "clean_columns_before_final_column_selection",
            "duplicate_rows_removed",
            "bad_null_rows",
            "invalid_allow_register_rows",
            "remaining_logical_key_duplicates",
        ],
        "value": [
            raw_row_count,
            len(df),
            raw_shape[1],
            df.shape[1],
            duplicate_rows_removed,
            len(bad_null_rows),
            len(remaining_invalid_allow_register),
            int(df.duplicated(logical_key).sum()),
        ],
    }
)

display(validation_summary)
print("Raw row count:", raw_row_count)
print("Clean row count:", len(df))
print("Rows removed due to duplicates:", duplicate_rows_removed)


Null report after cleaning:


,column,dtype,null_count,null_percent
0,allow_register,string,131249,79.6723
1,course_credits,Float64,0,0.0000
2,course_id,string,0,0.0000
3,course_name_sl,string,0,0.0000
4,course_type_id,Int64,0,0.0000
5,department_id,string,0,0.0000
6,department_name_sl,string,0,0.0000
7,faculty_id,string,0,0.0000
8,faculty_name_sl,string,0,0.0000
9,is_registration_allowed,boolean,0,0.0000


Bad null rows in critical non-allow_register columns: 0


,metric,value
0,raw_rows,164737
1,clean_rows_before_final_column_selection,164736
2,raw_columns,12
3,clean_columns_before_final_column_selection,14
4,duplicate_rows_removed,1
5,bad_null_rows,0
6,invalid_allow_register_rows,0
7,remaining_logical_key_duplicates,0


Raw row count: 164737
Clean row count: 164736
Rows removed due to duplicates: 1


In [12]:
# Cell 10: final output DataFrames
# Keep required business columns plus any project-standard helper columns if they exist.
id_helper_output_columns = [
    column
    for column in df.columns
    if column.startswith(("department_id_", "faculty_id_", "course_id_"))
]

final_columns = final_business_columns + [
    column for column in id_helper_output_columns if column not in final_business_columns
]

df_clean_course_offer = (
    df[final_columns]
    .sort_values(logical_key)
    .reset_index(drop=True)
    .copy()
)

df_available_course_offer = df_clean_course_offer.loc[
    df_clean_course_offer["is_registration_allowed"]
].copy()

print("df_clean_course_offer shape:", df_clean_course_offer.shape)
print("df_available_course_offer shape:", df_available_course_offer.shape)
print("Final columns:", df_clean_course_offer.columns.tolist())

print("Available-course filter example:")
print('available_courses_for_part = df_clean_course_offer[(df_clean_course_offer["part_id"] == target_part_id) & (df_clean_course_offer["is_registration_allowed"])]')

display(df_clean_course_offer.head())
display(df_available_course_offer.head())


df_clean_course_offer shape: (164736, 12)
df_available_course_offer shape: (30133, 12)
Final columns: ['part_id', 'faculty_id', 'department_id', 'level_category_id', 'course_id', 'course_type_id', 'course_credits', 'allow_register', 'is_registration_allowed', 'course_name_sl', 'faculty_name_sl', 'department_name_sl']
Available-course filter example:
available_courses_for_part = df_clean_course_offer[(df_clean_course_offer["part_id"] == target_part_id) & (df_clean_course_offer["is_registration_allowed"])]


,part_id,faculty_id,department_id,level_category_id,course_id,course_type_id,course_credits,allow_register,is_registration_allowed,course_name_sl,faculty_name_sl,department_name_sl
0,20050,167.111,11.111,1,1171.111,3,3.0,<NA>,False,مدخل إلى الخوارزميات والبرمجة,كلية الهندسة,قسم مقررات كلية الهندسة
1,20050,167.111,11.111,1,1172.111,3,3.0,<NA>,False,الرياضيات المتقطعة,كلية الهندسة,قسم مقررات كلية الهندسة
2,20050,167.111,11.111,1,1173.111,3,3.0,<NA>,False,الجبر الخطي ونظرية المصفوفات,كلية الهندسة,قسم مقررات كلية الهندسة
3,20050,167.111,11.111,1,1174.111,3,3.0,<NA>,False,الفيزياء1,كلية الهندسة,قسم مقررات كلية الهندسة
4,20050,167.111,11.111,1,1175.111,3,3.0,<NA>,False,التحليل الرياضي1,كلية الهندسة,قسم مقررات كلية الهندسة


,part_id,faculty_id,department_id,level_category_id,course_id,course_type_id,course_credits,allow_register,is_registration_allowed,course_name_sl,faculty_name_sl,department_name_sl
2216,20051,2.111,8.111,1,562.111,3,3.0,Y,True,الفيزياء الطبية (1),كلية الطب البشري,قسم مقررات كلية الطب البشري
2217,20051,2.111,8.111,1,563.111,3,3.0,Y,True,علم الخلية,كلية الطب البشري,قسم مقررات كلية الطب البشري
2218,20051,2.111,8.111,1,564.111,3,3.0,Y,True,علم الحياة,كلية الطب البشري,قسم مقررات كلية الطب البشري
2219,20051,2.111,8.111,1,565.111,3,3.0,Y,True,الكيمياء العامة,كلية الطب البشري,قسم مقررات كلية الطب البشري
2384,20051,3.111,9.111,1,211.111,3,3.0,Y,True,النبات والوراثة,كلية طب الأسنان,قسم مقررات كلية طب الأسنان
